In [1]:
%pip install torch torchvision matplotlib seaborn scikit-learn kaggle


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import zipfile

# Setup kaggle token directory
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download Intel Image Classification dataset
!kaggle datasets download -d puneet6060/intel-image-classification

# Extract the archive
with zipfile.ZipFile("intel-image-classification.zip", "r") as zip_ref:
    zip_ref.extractall("intel_dataset")

print("Dataset ready!")

cp: kaggle.json: No such file or directory
chmod: /Users/abinraja/.kaggle/kaggle.json: No such file or directory
Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
intel-image-classification.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset ready!


In [3]:
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Hardware acceleration setup (MPS for Mac Apple Silicon, CUDA for Nvidia, CPU fallback)
if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print(f"Using device: {device}")

# Preprocessing & Data Augmentation pipelines (150x150, normalized)
train_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(150, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

test_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

# Load Datasets
full_train_dataset = datasets.ImageFolder(
    "intel_dataset/seg_train/seg_train", transform=train_transform
)
test_dataset = datasets.ImageFolder(
    "intel_dataset/seg_test/seg_test", transform=test_transform
)

class_names = full_train_dataset.classes
print(f"Detected Classes ({len(class_names)}): {class_names}")

# Train/Validation Split (80% Train, 20% Val)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

# DataLoader Pipeline
BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(
    f"Samples -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}"
)

Using device: mps
Detected Classes (6): ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Samples -> Train: 11227, Val: 2807, Test: 3000


In [ ]:
# Define Custom CNN Architecture (4 Conv Blocks + MaxPool + Dropout + FC Layers)
class CNNModel(nn.Module):

  def __init__(self, num_classes=6):
    super(CNNModel, self).__init__()
    self.features = nn.Sequential(
        # Block 1
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # Output: 32 x 75 x 75
        # Block 2
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # Output: 64 x 37 x 37
        # Block 3
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # Output: 128 x 18 x 18
        # Block 4
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # Output: 128 x 9 x 9
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128 * 9 * 9, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes),
    )

  def forward(self, x):
    return self.classifier(self.features(x))


model = CNNModel(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop with Early Stopping / Model Checkpointing
EPOCHS = 15
best_val_loss = float("inf")
patience = 4
patience_counter = 0

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
  # Training Phase
  model.train()
  running_loss, running_corrects = 0.0, 0
  for inputs, labels in train_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * inputs.size(0)
    running_corrects += (outputs.argmax(1) == labels).sum().item()

  epoch_train_loss = running_loss / len(train_dataset)
  epoch_train_acc = running_corrects / len(train_dataset)

  # Validation Phase
  model.eval()
  val_loss, val_corrects = 0.0, 0
  with torch.no_grad():
    for inputs, labels in val_loader:
      inputs, labels = inputs.to(device), labels.to(device)
      outputs = model(inputs)
      loss = criterion(outputs, labels)

      val_loss += loss.item() * inputs.size(0)
      val_corrects += (outputs.argmax(1) == labels).sum().item()

  epoch_val_loss = val_loss / len(val_dataset)
  epoch_val_acc = val_corrects / len(val_dataset)

  history["train_loss"].append(epoch_train_loss)
  history["train_acc"].append(epoch_train_acc)
  history["val_loss"].append(epoch_val_loss)
  history["val_acc"].append(epoch_val_acc)

  print(
      f"Epoch [{epoch+1}/{EPOCHS}] | "
      f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
      f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}"
  )

  # Checkpoint & Early Stopping Check
  if epoch_val_loss < best_val_loss:
    best_val_loss = epoch_val_loss
    torch.save(model.state_dict(), "best_cnn_model.pth")
    patience_counter = 0
  else:
    patience_counter += 1
    if patience_counter >= patience:
      print(f"Early stopping triggered at epoch {epoch+1}")
      break

# Load the best saved weights
model.load_state_dict(torch.load("best_cnn_model.pth"))

Epoch [1/15] | Train Loss: 1.0649, Train Acc: 0.5849 | Val Loss: 0.8325, Val Acc: 0.6783
Epoch [2/15] | Train Loss: 0.7653, Train Acc: 0.7225 | Val Loss: 0.6585, Val Acc: 0.7556


In [ ]:
# Plotting Loss & Accuracy curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Plot
ax1.plot(history["train_acc"], label="Train Accuracy", marker="o")
ax1.plot(history["val_acc"], label="Validation Accuracy", marker="o")
ax1.set_title("Accuracy Curves")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

# Loss Plot
ax2.plot(history["train_loss"], label="Train Loss", marker="o")
ax2.plot(history["val_loss"], label="Validation Loss", marker="o")
ax2.set_title("Loss Curves")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on Test Set
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
  for inputs, labels in test_loader:
    inputs = inputs.to(device)
    outputs = model(inputs)
    preds = outputs.argmax(1).cpu().numpy()

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Test Accuracy
test_accuracy = (y_true == y_pred).mean() * 100
print(f"Overall Test Accuracy: {test_accuracy:.2f}%\n")

# Classification Report (Precision, Recall, F1-Score)
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Evaluate on Test Set
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
  for inputs, labels in test_loader:
    inputs = inputs.to(device)
    outputs = model(inputs)
    preds = outputs.argmax(1).cpu().numpy()

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Test Accuracy
test_accuracy = (y_true == y_pred).mean() * 100
print(f"Overall Test Accuracy: {test_accuracy:.2f}%\n")

# Classification Report (Precision, Recall, F1-Score)
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Denormalize helper function for image display
def denormalize(tensor):
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  img = tensor.permute(1, 2, 0).cpu().numpy()
  img = std * img + mean
  return np.clip(img, 0, 1)


# Plot 10 sample predictions
data_iter = iter(test_loader)
images, labels = next(data_iter)

model.eval()
with torch.no_grad():
  outputs = model(images.to(device))
  preds = outputs.argmax(1).cpu().numpy()

plt.figure(figsize=(16, 7))
for i in range(10):
  ax = plt.subplot(2, 5, i + 1)
  img_disp = denormalize(images[i])
  plt.imshow(img_disp)

  actual_label = class_names[labels[i].item()]
  pred_label = class_names[preds[i]]
  color = "green" if pred_label == actual_label else "red"

  plt.title(f"Pred: {pred_label}\nTrue: {actual_label}", color=color, fontsize=10)
  plt.axis("off")

plt.tight_layout()
plt.show()